In [ ]:
import torch
import random
import numpy as np
import pandas as pd
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSeq2SeqLM, TrainingArguments,Trainer, AutoTokenizer, DataCollatorForSeq2Seq

In [ ]:
def scramble_sentence(sentence, shuffle_prob=0.9, max_shuffle_window=5):
    tokens = sentence.split()

    if random.random() < shuffle_prob and len(tokens) > 2:
        new_tokens = tokens[:]
        for i in range(len(tokens)):
            if random.random() < 0.75:
                j = min(len(tokens)-1, i + random.randint(1, max_shuffle_window))
                new_tokens[i], new_tokens[j] = new_tokens[j], new_tokens[i]
        tokens = new_tokens
    return " ".join(tokens)

In [ ]:
def build_dataset(sl_sentences, num_variants=5):
    inputs = []
    targets = []

    for sent in sl_sentences:
        for _ in range(num_variants):
            noisy = scramble_sentence(sent)
            inputs.append(noisy)
            targets.append(sent)

    return inputs, targets

In [ ]:
with open('/kaggle/input/arsl-tr01-glossed-processed/ArSL-tr01-glossedProcessed.txt','r',encoding='utf-8') as f:
    sl_sentences = [
        line.strip()
        for line in f
        if line.strip()
    ]

inputs, targets = build_dataset(sl_sentences, num_variants=5)

dataset = Dataset.from_dict({
    "input_text": inputs,
    "target_text": targets
})

dataset = dataset.train_test_split(test_size=0.1)

In [ ]:
dataset['train']['input_text']

In [ ]:
dataset['train']['input_text']

In [ ]:
dataset['test']['input_text']

In [ ]:
#MODEL_NAME = "UBC-NLP/AraT5-base"
MODEL_NAME = "UBC-NLP/AraT5v2-base-1024"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 64

def tokenize(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

    labels = tokenizer(
        batch["target_text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

    model_inputs["labels"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]]

    return model_inputs

tokenized = dataset.map(tokenize, batched=True, remove_columns=dataset["train"].column_names)

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
data_collator = DataCollatorForSeq2Seq(tokenizer, model = model)

training_args = TrainingArguments(
    output_dir="./sl_reorder",
    eval_strategy="steps",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    num_train_epochs=5,
    warmup_steps=500,
    logging_steps=30,
    save_steps=500,
    save_total_limit=2,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    data_collator = data_collator
)

trainer.train()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
def reorder_to_sl(sentence):
    inputs = tokenizer(sentence, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_length=64,
        num_beams=5
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(reorder_to_sl(""))

In [108]:
import torchinfo
torchinfo.summary(model)

Layer (type:depth-idx)                                       Param #
T5ForConditionalGeneration                                   --
├─Embedding: 1-1                                             84,639,744
├─T5Stack: 1-2                                               84,639,744
│    └─Embedding: 2-1                                        (recursive)
│    └─ModuleList: 2-2                                       --
│    │    └─T5Block: 3-1                                     7,079,808
│    │    └─T5Block: 3-2                                     7,079,424
│    │    └─T5Block: 3-3                                     7,079,424
│    │    └─T5Block: 3-4                                     7,079,424
│    │    └─T5Block: 3-5                                     7,079,424
│    │    └─T5Block: 3-6                                     7,079,424
│    │    └─T5Block: 3-7                                     7,079,424
│    │    └─T5Block: 3-8                                     7,079,424
│    │    └─T5Bloc

In [ ]:
model.save_pretrained("/kaggle/working/AraT5v2")
tokenizer.save_pretrained("/kaggle/working/AraT5v2")

In [103]:
def generate_predictions(sentences, batch_size=16):
    model.eval()
    preds = []

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LEN
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=MAX_LEN,
                num_beams=5
            )

        decoded = tokenizer.batch_decode(
            outputs, skip_special_tokens=True
        )
        preds.extend(decoded)

    return preds


In [104]:
raw_test = dataset["test"]
test_inputs = raw_test["input_text"]
test_targets = raw_test["target_text"]

test_preds = generate_predictions(test_inputs)

In [ ]:
for i in range(len(test_inputs)):
    print("="*80)
    print(f"[{i}] INPUT (scrambled):")
    print(test_inputs[i])
    print("\nTARGET (gold SL order):")
    print(test_targets[i])
    print("\nPREDICTED:")
    print(test_preds[i])


In [ ]:
exact_match = np.mean([
    p.strip() == t.strip()
    for p, t in zip(test_preds, test_targets)
])

print(f"Exact Match Accuracy: {exact_match:.4f}")

In [107]:
df = pd.DataFrame({
    "input_scrambled": test_inputs,
    "target_sl": test_targets,
    "predicted_sl": test_preds
})

df.to_csv("test_predictions.csv", index=False, encoding="utf-8-sig")
